In [24]:
!pip install category_encoders


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import category_encoders as ce
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
import warnings
# Tắt riêng cảnh báo về validation feature names của sklearn
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.validation")

In [26]:
import pandas as pd

df_4_6 = pd.read_csv('gold_4_6.csv')
df_7_9 = pd.read_csv('gold_7_9.csv')

In [27]:
df_4_6.columns

Index(['Order date', 'CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'INNER_CD',
       'SUPPLIER_CD', 'Sales order line number',
       'Consider count hodiday Saturday', 'OTHER AREA SHIP DIV',
       'ALLOCATION QTY', 'PACKING RANK', 'PURCHASE AMOUNT', 'VSD',
       'DIRECT SHIP FLG', 'DELI_DIV', 'label', 'Ship Mode', 'PACK QTY',
       'WEIGHT PER PIECE', 'SPECIAL_DIV', 'SO_DAY_OF_MONTH', 'SO_DAY_OF_WEEK',
       'SO_TIME', 'SO_TIME_str', 'hour', 'time_period', 'IS_WEEKEND',
       'MONTH_PHASE', 'Order_month', 'VSD_month', 'VSD_IS_WEEKEND',
       'VSD_MONTH_PHASE', 'Expected_delivery_days'],
      dtype='str')

'CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'INNER_CD',
       'SUPPLIER_CD' - Hash encode

In [28]:
df_7_9.columns

Index(['Order date', 'CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'INNER_CD',
       'SUPPLIER_CD', 'Sales order line number',
       'Consider count hodiday Saturday', 'OTHER AREA SHIP DIV',
       'ALLOCATION QTY', 'SUPPLIER INV AMOUNT', 'PACKING RANK', 'VSD',
       'DIRECT SHIP FLG', 'DELI_DIV', 'label', 'Ship Mode', 'PACK QTY',
       'WEIGHT PER PIECE', 'SPECIAL_DIV', 'SO_DAY_OF_MONTH', 'SO_DAY_OF_WEEK',
       'SO_TIME', 'SO_TIME_str', 'hour', 'time_period', 'IS_WEEKEND',
       'MONTH_PHASE', 'Order_month', 'VSD_month', 'VSD_IS_WEEKEND',
       'VSD_MONTH_PHASE', 'Expected_delivery_days'],
      dtype='str')

In [29]:
# Select columns have dtype = object, str
object_cols = df_4_6.select_dtypes(include=['object', 'str']).columns.tolist()

print("Object columns:")
print(object_cols)

Object columns:
['Order date', 'BRAND_CD', 'INNER_CD', 'SUPPLIER_CD', 'OTHER AREA SHIP DIV', 'PACKING RANK', 'VSD', 'DELI_DIV', 'Ship Mode', 'time_period', 'MONTH_PHASE', 'VSD_MONTH_PHASE']


# Chia theo train - dev - test

In [30]:
# Chia tập A
X_A = df_4_6.drop(columns=['label','Order date', 'VSD'])
y_A = df_4_6['label']

X_A_train, X_A_test, y_A_train, y_A_test = train_test_split(X_A, y_A, test_size=0.2, random_state=42, stratify=y_A)

In [31]:
X_B = df_7_9.drop(columns=['label','Order date', 'VSD'])
y_B = df_7_9['label']

X_B_train, X_B_test, y_B_train, y_B_test = train_test_split(X_B, y_B, test_size=0.2, random_state=42, stratify=y_B)

In [32]:
target_encode = ['CLASSIFY_CD', 'CUST_CD', 'BRAND_CD', 'SUPPLIER_CD']
one_hot_encode = [
    'Consider count hodiday Saturday', 'OTHER AREA SHIP DIV', 'PACKING RANK', 'DIRECT SHIP FLG',
    'DELI_DIV', 'Ship Mode', 'WEIGHT PER PIECE',
    'SPECIAL_DIV', 'time_period', 'IS_WEEKEND', 'MONTH_PHASE',
    'Order_month', 'VSD_month', 'VSD_IS_WEEKEND', 'VSD_MONTH_PHASE'
]

In [33]:
preprocessor_A = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)

preprocessor_B = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)

# Undersampling majority (majority : minority = 4 : 1)
undersample = RandomUnderSampler(
    sampling_strategy=0.5,
    random_state=42
)
smote = SMOTE(sampling_strategy=0.25, random_state=42 )
pipeline_A = ImbPipeline(steps=[
    ('preprocess', preprocessor_A),
    ('smote', smote),
    ('undersample', undersample),
])

pipeline_B = ImbPipeline(steps=[
    ('preprocess', preprocessor_B),
    ('smote', smote),
    ('undersample', undersample),
])
X_A_resampled, y_A_resampled = pipeline_A.fit_resample(X_A_train, y_A_train)
X_B_resampled, y_B_resampled = pipeline_B.fit_resample(X_B_train, y_B_train)
print("Before sampling:")
print(y_A_train.value_counts())

print("\nAfter sampling:")
print(pd.Series(y_A_resampled).value_counts())


Before sampling:
label
0    311453
1      7786
Name: count, dtype: int64

After sampling:
label
0    155726
1     77863
Name: count, dtype: int64


In [34]:
X_A_test_processed = pipeline_A.named_steps['preprocess'].transform(X_A_test)
X_B_test_processed = pipeline_B.named_steps['preprocess'].transform(X_B_test)

In [35]:
# Làm sạch dữ liệu trước khi cho vào Pipeline
for col in one_hot_encode:
    # Tập A
    X_A_train[col] = X_A_train[col].fillna('Unknown').astype(str)
    X_A_test[col] = X_A_test[col].fillna('Unknown').astype(str)
    # Tập B
    X_B_train[col] = X_B_train[col].fillna('Unknown').astype(str)
    X_B_test[col] = X_B_test[col].fillna('Unknown').astype(str)

In [36]:
X_B_test_processed1 = pipeline_A.named_steps['preprocess'].transform(X_B_test)
X_A_test_processed1 = pipeline_B.named_steps['preprocess'].transform(X_A_test)

In [37]:
neg = y_A_resampled.value_counts()[0]
pos = y_A_resampled.value_counts()[1]

In [38]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_A_resampled, y_A_resampled)

y_A_pred_knn = knn.predict(X_A_test_processed)
y_A_prob_knn = knn.predict_proba(X_A_test_processed)[:, 1]

print("==== KNN ====")
print("Confusion Matrix:")
print(confusion_matrix(y_A_test, y_A_pred_knn))

print("\nClassification Report:")
print(classification_report(y_A_test, y_A_pred_knn, digits=4))

auc_knn = roc_auc_score(y_A_test, y_A_prob_knn)
print(f"AUC-ROC: {auc_knn:.4f}")

==== KNN ====
Confusion Matrix:
[[59538 18325]
 [  332  1615]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9945    0.7647    0.8645     77863
           1     0.0810    0.8295    0.1476      1947

    accuracy                         0.7662     79810
   macro avg     0.5377    0.7971    0.5061     79810
weighted avg     0.9722    0.7662    0.8471     79810

AUC-ROC: 0.8479


In [39]:
neg = y_A_train.value_counts()[0]
pos = y_A_train.value_counts()[1]

xgb_model = XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric='aucpr',
    scale_pos_weight=neg/pos
)

xgb_model.fit(X_A_resampled, y_A_resampled)

y_A_pred_xgb= xgb_model.predict(X_A_test_processed)
y_A_prob_xgb = xgb_model.predict_proba(X_A_test_processed)[:, 1]

print("==== XGBoost ====")
print("Confusion Matrix:")
print(confusion_matrix(y_A_test, y_A_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_A_test, y_A_pred_xgb, digits=4))

auc_xgb = roc_auc_score(y_A_test, y_A_prob_xgb)
print(f"AUC-ROC: {auc_xgb:.4f}")

==== XGBoost ====
Confusion Matrix:
[[64550 13313]
 [  237  1710]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9963    0.8290    0.9050     77863
           1     0.1138    0.8783    0.2015      1947

    accuracy                         0.8302     79810
   macro avg     0.5551    0.8536    0.5533     79810
weighted avg     0.9748    0.8302    0.8879     79810

AUC-ROC: 0.9348


In [40]:
lgbm_model = LGBMClassifier(
    objective='binary',
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=35,
    random_state=42,
    scale_pos_weight=neg/pos
)
lgbm_model.fit(X_A_resampled, y_A_resampled)

y_A_pred_lgbm = lgbm_model.predict(X_A_test_processed)
y_A_prob_lgbm = lgbm_model.predict_proba(X_A_test_processed)[:, 1]

print("==== LightGBM====")
print("Confusion Matrix:")
print(confusion_matrix(y_A_test, y_A_pred_lgbm))

print("\nClassification Report:")
print(classification_report(y_A_test, y_A_pred_lgbm, digits=4))

auc_lgbm = roc_auc_score(y_A_test, y_A_prob_lgbm)
print(f"AUC-ROC: {auc_lgbm:.4f}")

[LightGBM] [Info] Number of positive: 77863, number of negative: 155726
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.127459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39254
[LightGBM] [Info] Number of data points in the train set: 233589, number of used features: 1069
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
==== LightGBM====
Confusion Matrix:
[[67899  9964]
 [  242  1705]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9964    0.8720    0.9301     77863
           1     0.1461    0.8757    0.2504      1947

    accuracy                         0.8721     79810
   macro avg     0.5713    0.8739    0.5903     79810
weighted avg     0.9757    0.8721    0.9135     79810

AUC-ROC: 0.9479


# Chia theo K-Fold

In [41]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, make_scorer

knn = KNeighborsClassifier()

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9]
}

f1_scorer = make_scorer(f1_score, pos_label=1)

grid_knn = GridSearchCV(
    estimator=knn,
    param_grid=param_grid_knn,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for KNN")
grid_knn.fit(X_A_resampled, y_A_resampled)

print("Best parameters (KNN):", grid_knn.best_params_)
best_knn_params = grid_knn.best_params_
best_knn_model = grid_knn.best_estimator_

print("Best CV score (mean f1):", round(grid_knn.best_score_, 4))

y_kfold_pred_knn = best_knn_model.predict(X_A_test_processed)
y_kfold_prob_knn = best_knn_model.predict_proba(X_A_test_processed)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_knn, zero_division=0, digits = 4))
auc_knn_kfold = roc_auc_score(y_A_test, y_kfold_prob_knn)
print(f"AUC-ROC: {auc_knn_kfold:.4f}")

Start training process for KNN
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best parameters (KNN): {'n_neighbors': 3}
Best CV score (mean f1): 0.8104

Classification Report:
              precision    recall  f1-score   support

           0     0.9942    0.7915    0.8814     77863
           1     0.0892    0.8161    0.1608      1947

    accuracy                         0.7921     79810
   macro avg     0.5417    0.8038    0.5211     79810
weighted avg     0.9721    0.7921    0.8638     79810

AUC-ROC: 0.8320


In [42]:
xgb_base = XGBClassifier(eval_metric='auc', random_state=42, scale_pos_weight=neg/pos)

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7, 10],
    'learning_rate': [0.05, 0.1],
}

grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for XGBoost")
grid_xgb.fit(X_A_resampled, y_A_resampled)

print("Best parameters (XGBoost):", grid_xgb.best_params_)
best_xgb_params = grid_xgb.best_params_
best_xgb_model = grid_xgb.best_estimator_

print("Best CV score (mean f1):", round(grid_xgb.best_score_, 4))

y_kfold_pred_xgb= best_xgb_model.predict(X_A_test_processed)
y_kfold_prob_xgb = best_xgb_model.predict_proba(X_A_test_processed)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_xgb, zero_division=0, digits = 4))
auc_xgb_kfold = roc_auc_score(y_A_test, y_kfold_prob_xgb)
print(f"AUC-ROC: {auc_xgb_kfold:.4f}")


Start training process for XGBoost
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters (XGBoost): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200}
Best CV score (mean f1): 0.9518

Classification Report:
              precision    recall  f1-score   support

           0     0.9945    0.9426    0.9679     77863
           1     0.2565    0.7915    0.3874      1947

    accuracy                         0.9389     79810
   macro avg     0.6255    0.8671    0.6776     79810
weighted avg     0.9765    0.9389    0.9537     79810

AUC-ROC: 0.9426


In [43]:
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_class_weight

lgbm = LGBMClassifier(objective='binary', random_state=42)

param_grid_lgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7, 10],
    'num_leaves': [15, 31]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgb,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for LightGBM")
grid_lgbm.fit(X_A_resampled, y_A_resampled)

print("Best parameters (LightGBM):", grid_lgbm.best_params_)
best_lgbm_params = grid_lgbm.best_params_
best_lgbm_model = grid_lgbm.best_estimator_

print("Best CV score (mean f1):", round(grid_lgbm.best_score_, 4))

y_kfold_pred_lgbm= best_lgbm_model.predict(X_A_test_processed)
y_kfold_prob_lgbm = best_lgbm_model.predict_proba(X_A_test_processed)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_lgbm, zero_division=0, digits = 4))
auc_lgbm_kfold = roc_auc_score(y_A_test, y_kfold_prob_lgbm)
print(f"AUC-ROC: {auc_lgbm_kfold:.4f}")

Start training process for LightGBM
Fitting 3 folds for each of 24 candidates, totalling 72 fits
[LightGBM] [Info] Number of positive: 77863, number of negative: 155726
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.160948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39254
[LightGBM] [Info] Number of data points in the train set: 233589, number of used features: 1069
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Best parameters (LightGBM): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Best CV score (mean f1): 0.9725

Classification Report:
              precision    recall  f1-score   support

           0     0.9910    0.9845    0.9877    

# Train trên A, test trên B

In [44]:
y_kfold_pred_lgbm_B= best_lgbm_model.predict(X_B_test_processed1)
y_kfold_prob_lgbm_B = best_lgbm_model.predict_proba(X_B_test_processed1)[:, 1]
print("\nClassification Report:")
print(classification_report(y_B_test, y_kfold_pred_lgbm_B, zero_division=0, digits = 4))
auc_lgbm_kfold_B = roc_auc_score(y_B_test, y_kfold_prob_lgbm_B)
print(f"AUC-ROC: {auc_lgbm_kfold_B:.4f}")


Classification Report:
              precision    recall  f1-score   support

           0     0.9760    0.9923    0.9841    209737
           1     0.0771    0.0257    0.0386      5243

    accuracy                         0.9687    214980
   macro avg     0.5265    0.5090    0.5113    214980
weighted avg     0.9541    0.9687    0.9610    214980

AUC-ROC: 0.6757


# Train trên B, test trên A

In [45]:
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_class_weight

lgbm = LGBMClassifier(objective='binary', random_state=42)

param_grid_lgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7, 10],
    'num_leaves': [15, 31]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgb,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for LightGBM")
grid_lgbm.fit(X_B_resampled, y_B_resampled)

print("Best parameters (LightGBM):", grid_lgbm.best_params_)
best_lgbm_params = grid_lgbm.best_params_
best_lgbm_model = grid_lgbm.best_estimator_

print("Best CV score (mean f1):", round(grid_lgbm.best_score_, 4))

y_kfold_pred_lgbm_A= best_lgbm_model.predict(X_A_test_processed1)
y_kfold_prob_lgbm_A = best_lgbm_model.predict_proba(X_A_test_processed1)[:, 1]
print("\nClassification Report:")
print(classification_report(y_A_test, y_kfold_pred_lgbm_A, zero_division=0, digits = 4))
auc_lgbm_kfold = roc_auc_score(y_A_test, y_kfold_prob_lgbm_A)
print(f"AUC-ROC: {auc_lgbm_kfold:.4f}")

Start training process for LightGBM
Fitting 3 folds for each of 24 candidates, totalling 72 fits
[LightGBM] [Info] Number of positive: 209735, number of negative: 419470
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.594073 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 36498
[LightGBM] [Info] Number of data points in the train set: 629205, number of used features: 1163
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
Best parameters (LightGBM): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Best CV score (mean f1): 0.965

Classification Report:
              precision    recall  f1-score   support

           0     0.9770    0.9936    0.9852     77863
           1     0.1955    0.0627    0.0949      1947

    accuracy 

# Train trên A+B, kfold ###

In [46]:
X_raw_train = pd.concat([X_A_train, X_B_train]).reset_index(drop=True)
y_raw_train = pd.concat([y_A_train, y_B_train]).reset_index(drop=True)

X_raw_test = pd.concat([X_A_test, X_B_test]).reset_index(drop=True)
y_raw_test = pd.concat([y_A_test, y_B_test]).reset_index(drop=True)
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)

pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(sampling_strategy=0.25, random_state=42)),
    ('undersample', RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
])

X_resampled, y_resampled = pipeline.fit_resample(X_raw_train, y_raw_train)
X_test_processed = pipeline.named_steps['preprocess'].transform(X_raw_test)

In [47]:
lgbm = LGBMClassifier(objective='binary', random_state=42)

param_grid_lgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7, 10],
    'num_leaves': [15, 31]
}

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid_lgb,
    scoring=f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)
print(f"Start training process for LightGBM")
grid_lgbm.fit(X_resampled, y_resampled)

print("Best parameters (XGBoost):", grid_lgbm.best_params_)
best_lgbm_params = grid_lgbm.best_params_
best_lgbm_model = grid_lgbm.best_estimator_

print("Best CV score (mean f1):", round(grid_lgbm.best_score_, 4))

y_pred= best_lgbm_model.predict(X_test_processed )
y_prob = best_lgbm_model.predict_proba(X_test_processed )[:, 1]
print("\nClassification Report:")
print(classification_report(y_raw_test, y_pred, zero_division=0, digits = 4))
auc = roc_auc_score(y_raw_test, y_prob)
print(f"AUC-ROC: {auc:.4f}")

Start training process for LightGBM
Fitting 3 folds for each of 24 candidates, totalling 72 fits
[LightGBM] [Info] Number of positive: 287599, number of negative: 575198
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.606684 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 39684
[LightGBM] [Info] Number of data points in the train set: 862797, number of used features: 1204
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147
Best parameters (XGBoost): {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 200, 'num_leaves': 31}
Best CV score (mean f1): 0.9573

Classification Report:
              precision    recall  f1-score   support

           0     0.9920    0.9819    0.9870    287600
           1     0.4865    0.6846    0.5688      7190

    accuracy                         0.9747    294790
   macro avg     0.7392

# Train trên A+KB, Test trên (1-K)B

In [48]:
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', ce.TargetEncoder(cols=target_encode), target_encode),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True), one_hot_encode)
    ],
    remainder='drop'
)
from sklearn.model_selection import train_test_split
from scipy.sparse import vstack

K_values = [30, 50, 70]

for K in K_values:
    print(f"\n=== K = {K}% ===")

    X_KB_raw, X_not_KB_raw, y_KB_raw, y_not_KB_raw = train_test_split(
        X_B_train, y_B_train,
        train_size=K/100,
        stratify=y_B_train,
        random_state=42
    )

    X_train_raw = pd.concat([X_A_train, X_KB_raw]).reset_index(drop=True)
    y_train_raw = pd.concat([y_A_train, y_KB_raw]).reset_index(drop=True)

    pipeline = ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('smote', SMOTE(sampling_strategy=0.25, random_state=42)),
        ('undersample', RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
        ('model', LGBMClassifier(
            objective='binary',
            random_state=42,
            learning_rate=0.1,
            max_depth=10,
            n_estimators=200,
            num_leaves=31
        ))
    ])

    pipeline.fit(X_train_raw, y_train_raw)

    y_pred = pipeline.predict(X_not_KB_raw)
    y_prob = pipeline.predict_proba(X_not_KB_raw)[:, 1]

    print("\nClassification Report:")
    print(classification_report(y_not_KB_raw, y_pred, zero_division=0, digits=4))

    auc = roc_auc_score(y_not_KB_raw, y_prob)
    print(f"AUC-ROC: {auc:.4f}")


=== K = 30% ===
[LightGBM] [Info] Number of positive: 140784, number of negative: 281568
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.178900 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 42359
[LightGBM] [Info] Number of data points in the train set: 422352, number of used features: 1172
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.333333 -> initscore=-0.693147
[LightGBM] [Info] Start training from score -0.693147

Classification Report:
              precision    recall  f1-score   support

           0     0.9909    0.9877    0.9893    587260
           1     0.5642    0.6370    0.5984     14682

    accuracy                         0.9791    601942
   macro avg     0.7775    0.8123    0.7938    601942
weighted avg     0.9805    0.9791    0.9798    601942

AUC-ROC: 0.9308

=== K = 50% ===
[LightGBM] [Info] Number of positiv